# `inference_v2.reader` — как выгружать данные

Читалка объединяет то, что лежит в разных местах: пять источников HDF5 с хитами,
файлы вероятностей sig-noise рядом с каждым, каталог событий и десятки каталогов
предсказаний. Она **ничего не пишет на диск** — ни кэшей, ни промежуточных таблиц.

Три уровня, от дешёвого к дорогому:

| уровень | что даёт | что читает |
|---|---|---|
| `runs()`, `sources()` | что вообще есть | метаданные |
| `r.events(...)` | таблица на событие | DuckDB + мелкие массивы HDF5 |
| `r.stream(...)` | события **и хиты** | DuckDB + хиты из HDF5 |

Ниже — все функции выгрузки, все фильтры, и в конце замеры: как время зависит от
источника, размера выборки, набора фильтров и плотности отбора.

In [1]:
import os, sys, time
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
sys.path.insert(0, "/home/albert/Baikal2025")

import numpy as np
import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

from inference_v2 import reader

CHECKPOINT = ("260816_2250_da_nu_classifier_exp_full_E1_lambda0.01_FIXED_sn256"
              "@best_da_model")
print(reader.__doc__.split("Design")[0].strip()[:300])

Reading Baikal sources and model predictions, without materialising anything.

    from inference_v2 import reader

    reader.checkpoints()                # which scored checkpoints exist
    r = reader.open("exp_reco",
                    checkpoint="260816_2250_..._FIXED_sn256@best_da_model")

  


`checkpoints()` обходит каталоги предсказаний и читает `run_info.json` каждого.

Модельная сторона называется **checkpoint**, а не `run`, нарочно: в этом проекте
*run* — это прогон детектора. Он лежит колонкой в каталоге, полем в BARS и
частью каждого `part_key`, и он же приходит колонкой `run` из `r.events()`.
Одно слово в двух значениях в таблицах, на которые смотрят рядом, — верный
способ однажды перепутать.

| колонка | что означает |
|---|---|
| `sources` | какие источники этот чекпойнт проскорил (по числу баз в каталоге) |
| `threshold` | порог sig-noise, по которому отбирались хиты перед скорингом |
| `sn_batch` | размер батча sig-noise — он **определяет отбор хитов**, а не скорость |
| `info_sources` | о каких источниках вообще что-то записано в `run_info.json` |
| `training_resolves` | удалось ли пройти цепочку до обучающего датасета |

`threshold` берётся из имени файла базы (`mc_merged_thr0p8.duckdb` → 0.8) — это
единственная запись про порог, которая не может потеряться. Пустой он только у
префильтра, где база называется `_allhits`: там отбора по вероятности нет вовсе.

А вот `sn_batch` известен ровно **у одного прогона из 35**, и колонка `info_sources`
показывает почему. До второй версии схемы `run_info.json` был одним плоским словарём,
который каждый прогон скоринга перезаписывал целиком. Поэтому у 34 каталогов запись
описывает только тот источник, что скорился последним, — остальное затёрто
безвозвратно. Пустой `sn_batch` значит «не записано», а не «батча не было». Это
настоящий пробел: от размера батча зависит, какие хиты фильтр оставит.
Измерено: `run_info.json` описывает **все** свои источники лишь у 2
прогонов из 35, у остальных 32 — только часть.

In [2]:
known = reader.checkpoints()
print(f"проскоренных чекпойнтов: {len(known)}, "
      f"обучение разрешается у {known.training_resolves.sum()}")
known[known.checkpoint.str.contains("FIXED_sn256|E2_lambda|sngp")].head(8)

проскоренных чекпойнтов: 35, обучение разрешается у 33


,checkpoint,sources,threshold,sn_batch,info_sources,training_resolves,npy_dir
18,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
19,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
20,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
21,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
22,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
23,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
24,260705_0708_da_nu_classifier_exp_full_E2_lambd...,"exp_full, mc_merged",0.8,NaN,exp_full,True,nu_classifier_dataset_h5s0_thr0.8
32,260816_2250_da_nu_classifier_exp_full_E1_lambd...,"exp_full, exp_reco, mc_merged, mc_reco",0.8,256.0,"exp_full, exp_reco, mc_merged, mc_reco",True,nu_classifier_dataset_h5s0_thr0.8


Те, у кого цепочка не разрешается, перечислены отдельно с причиной. Просить у них
исключение обучающих событий нельзя — читалка не станет тихо пропускать фильтр, а
упадёт.

In [3]:
reader.unresolved()

,checkpoint,reason
0,260705_0702_da_nu_classifier_exp_full_E1_lambd...,no da_config.yaml beside the checkpoint (exper...
1,sngp_nu_classifier_baseline@best_sngp_model,no run_info.json in the directory


`sources()` печатает пять источников: какие группы HDF5 внутри, есть ли истина МК и
реконструкция, и **что в источнике сломано** — с измерением рядом с каждым
исключением, чтобы кат не переносили в следующий анализ вслепую.

In [4]:
print(reader.sources())

mc_merged  (baikal_mc_merged.h5)
  groups        muatm_2020, nuatm_2020, nue2_2020
  prime_prty    yes
  reco_prty     no
  [probs_coverage] sig-noise probabilities exist for 10,100 of the 20,004 HDF5 parts; the rest were never scored and simply do not appear.

exp_full  (exp_full.h5)
  groups        exp_full
  prime_prty    no
  reco_prty     no
  bad runs      part_s2020_c02_r0020, part_s2020_c02_r0249
  [bad_runs] channels 222/224/225/227 (one string of cluster 2) emit up to 3169.8 p.e. against a median hit of 0.97; 36-78% of their hits exceed 100 p.e. where a normal channel sits at 0.0066%. Events touching them score above 0.8 8.7x more often. These are the only two runs where those channels fire.
  [cluster_1_refuted] cluster 1 is not in exp_full at all, and the old cluster 1/4 exclusion was refuted -- do not reintroduce it.

mc_reco  (baikal_mc_reco.h5)
  groups        muatm, nuatm_conv, nuatm_prompt, nue2
  prime_prty    yes
  reco_prty     31
  fragment cut  n_gt_sig_hits > 5
 

## 2. Открыть читалку

`reader.open(source, run=...)` — источник плюс прогон предсказаний. Один объект
`Reader` читает один источник, размеченный одним чекпойнтом.

In [5]:
r = reader.open("exp_reco", checkpoint=CHECKPOINT)
print(r)
print(reader.SOURCES["exp_reco"].h5)
print(reader.SOURCES["exp_reco"].probs)

<Reader exp_reco @ 260816_2250_da_nu_classifier_exp_full_E1_lambda0.01_FIXED_sn256@best_da_model>
/home/albert/Baikal2025/data_manager/data/h5datasets/exp_reco.h5
/home/albert/Baikal2025/data_manager/data/h5datasets/exp_reco_probs_k_nsol_labelneq0_da_hs128_k0p0001.h5


## 3. Уровень 1: события без единого хита

`r.events(...)` возвращает таблицу с одной строкой на событие. Хиты не читаются
вовсе, поэтому это дёшево.

Что откуда берётся:

| колонка | источник |
|---|---|
| `event_fk` | глобальный ключ каталога, он же ключ базы предсказаний |
| `score`, `n_sn_hits`, `n_sn_strings` | таблица `predictions` прогона |
| `data_class`, `cluster`, `run` | `cat.events` |
| `part_key`, `local_idx` | `cat.h5_locations` — адрес строки в HDF5 |
| `reco_*` | `reco_prty` из HDF5 — **только там они и есть** |
| `prime_*` | `prime_prty` из HDF5, истина МК |

У reco-источников базы предсказаний содержат ровно четыре колонки, поэтому вся
реконструкция BARS приходит из HDF5, а не из базы.

In [6]:
ev = r.events(where=[reader.H8S3, "l.part_key = 'part_s2020_c04_r0074'"])
print(f"{len(ev):,} событий, {len(ev.columns)} колонок")
print("из базы и каталога:", [c for c in ev.columns if not c.startswith("reco_")])
print("из reco_prty:", [c for c in ev.columns if c.startswith("reco_")][:8], "...")
ev.head(3)

27,560 событий, 35 колонок
из базы и каталога: ['source', 'event_fk', 'data_class', 'cluster', 'run', 'part_key', 'local_idx', 'score', 'n_sn_hits', 'n_sn_strings']
из reco_prty: ['reco_thetaRec', 'reco_phiRec', 'reco_thetaErr', 'reco_phiErr', 'reco_funcValue', 'reco_timeChi2', 'reco_chargeTerm', 'reco_LLFit'] ...


,source,event_fk,data_class,cluster,run,part_key,local_idx,score,n_sn_hits,n_sn_strings,reco_thetaRec,reco_phiRec,reco_thetaErr,reco_phiErr,reco_funcValue,reco_timeChi2,reco_chargeTerm,reco_LLFit,reco_nHits,reco_nStrings,reco_nOMs,reco_pathLength,reco_timeXYZRec,reco_covMatrixStatus,reco_scfMaxTheta,reco_scfMinTheta,reco_scfTheta,reco_scfPhi,reco_pHit,reco_evCenterZ,reco_zDist,reco_nTriplets,reco_nCalls,reco_classBDT,reco_classBDTLowE
0,exp_reco,663729294,exp_reco,4,74,part_s2020_c04_r0074,0,0.001778,11,5,113.945633,148.110886,0.699632,0.579686,238.337555,222.202606,16.134949,0.0,9.0,4.0,9.0,177.258560,17522.425781,3.0,2.303834,1.989675,2.094395,2.932152,0.000075,339.113251,140.197037,1.0,69.0,-2.0,-2.0
1,exp_reco,663729295,exp_reco,4,74,part_s2020_c04_r0074,1,0.004234,10,5,102.093498,23.225554,1.556602,1.067121,9.367344,1.175260,8.192083,0.0,8.0,4.0,8.0,157.103241,16464.853516,3.0,2.408554,0.628319,1.780236,0.418879,0.000070,300.183075,76.402245,1.0,91.0,-2.0,-2.0
2,exp_reco,663729296,exp_reco,4,74,part_s2020_c04_r0074,2,0.000705,15,5,111.714905,60.964638,0.479322,0.832434,112.734833,102.161407,10.573425,0.0,12.0,5.0,12.0,168.365524,17886.193359,3.0,1.989675,1.675516,1.884956,0.942478,0.000276,464.810944,126.084991,3.0,85.0,-2.0,-2.0


## 4. Фильтры

Аргумент `where` принимает **список условий**, соединяемых через `AND`. В нём
смешиваются произвольный SQL и именованные предикаты.

Псевдонимы таблиц в SQL:

| псевдоним | таблица |
|---|---|
| `p` | `predictions` прогона — `score`, `n_sn_hits`, `n_sn_strings` |
| `e` | `cat.events` — `data_class`, `cluster`, `run`, `season`, `feature_hash` |
| `l` | `cat.h5_locations` — `part_key`, `local_idx`, `h5_path` |

**По умолчанию не режется ничего.** Это осознанно: молчаливый кат по умолчанию
однажды уже привёл к тому, что человек думал, будто смотрит на всё, а смотрел на
четверть.

In [7]:
# произвольный SQL
r.events(where=["p.score > 0.99", "e.cluster = 4",
                "l.part_key = 'part_s2020_c04_r0074'"])[
    ["event_fk", "cluster", "score", "n_sn_hits", "reco_thetaRec"]].head()

,event_fk,cluster,score,n_sn_hits,reco_thetaRec
0,663734597,4,0.993428,5,99.155571
1,663741087,4,0.995659,11,101.969551
2,663741536,4,0.990546,5,83.410110
3,663743196,4,0.991584,5,149.727814
4,663743389,4,0.992418,19,67.848808


### Именованные предикаты

| имя | что делает | чего требует |
|---|---|---|
| `reader.H8S3` | `n_sn_hits >= 8 AND n_sn_strings >= 3` | — |
| `reader.NOT_EXCLUDED` | убирает прогоны и кластеры с зафиксированными неисправностями | — |
| `reader.NOT_TRAINED` | убирает **всё**, что лежит в обучающем NPY-датасете | разрешимой цепочки |
| `reader.NOT_TRAINED_EXACT` | убирает только обучающую половину | того же |
| `reader.NOT_DA_TARGET` | убирает немеченую цель доменной адаптации | того же |

`NOT_EXCLUDED` зависит от источника: для `exp_reco` это кластер 1 и два прогона c02,
для `exp_full` — те же два прогона, для МК — ничего. Что именно и почему — в
`reader.sources()` выше.

In [8]:
# Кластер 1 в exp_reco исключается целиком: t_span 17 600 нс против 4 900,
# sig-noise находит там медианно НОЛЬ сигнальных хитов. Берём одну часть из
# кластера 1 и одну из кластера 4, чтобы увидеть, что именно снимается.
two = "l.part_key IN ('part_s2020_c01_r0017', 'part_s2020_c04_r0074')"
for label, conds in [
    ("без фильтров",   [two]),
    ("+ H8S3",         [two, reader.H8S3]),
    ("+ NOT_EXCLUDED", [two, reader.H8S3, reader.NOT_EXCLUDED]),
]:
    got = r.events(where=conds)
    by_cluster = got.cluster.value_counts().sort_index().to_dict()
    print(f"{label:<16} {len(got):>8,} событий   по кластерам: {by_cluster}")

без фильтров       50,450 событий   по кластерам: {1: 6751, 4: 43699}


+ H8S3             31,511 событий   по кластерам: {1: 3951, 4: 27560}


+ NOT_EXCLUDED     27,560 событий   по кластерам: {4: 27560}


### Исключение обучающих событий

Самый важный фильтр: скоры событий, на которых модель обучалась, смещены.

Читалка **не пользуется таблицей `splits`** — она есть у одного прогона из 35 и ей
нет доверия. Вместо этого она идёт к первоисточнику: `run_info.json` даёт чекпойнт,
чекпойнт даёт `da_config.yaml`, тот — каталог NPY-датасета, а датасет хранит
`(data_class, part_key, local_idx)` каждого своего события. Эта таблица
регистрируется в DuckDB и вычитается анти-джойном прямо в потоковом запросе.

Две строгости различаются тем, что датасет **шире**, чем то, что видело обучение:
`max_events` отбрасывает часть, `train_split` отдаёт часть в валидацию. По умолчанию
консервативно — убираем весь датасет, потому что валидационные события влияли на
выбор лучшей эпохи.

In [9]:
ckpt = reader.open_checkpoint(CHECKPOINT)
print("датасет:", ckpt.training_dataset().name)
print("отбор  :", ckpt.training_selection())

from inference_v2.reader import training
ident = training.identity(ckpt.training_dataset())
exact = training.identity(ckpt.training_dataset(), strictness="train",
                          selection=ckpt.training_selection())
print(f"\nвсего в датасете  : {len(ident):,}")
print(f"обучающая половина: {len(exact):,}")
ident.head(3)

датасет: nu_classifier_dataset_h5s0_thr0.8
отбор  : {'max_events': 5000000, 'train_split': 0.9, 'seed': 42}



всего в датасете  : 5,256,496
обучающая половина: 4,500,000


,data_class,part_key,local_idx
0,muatm_2020,part_12378,4166
1,muatm_2020,part_24465,22512
2,muatm_2020,part_37486,24906


In [10]:
# part_1087 класса nue2 отдала в обучение 5 228 событий и содержит 6 496
# проскоренных с h8s3 — на ней видно, что фильтр снимает.
rm = reader.open("mc_merged", checkpoint=CHECKPOINT)
nue = ["l.part_key = 'part_1087'", "e.data_class = 'nue2_2020'"]
for label, conds in [
    ("H8S3",                [*nue, reader.H8S3]),
    ("+ NOT_TRAINED",       [*nue, reader.H8S3, reader.NOT_TRAINED]),
    ("+ NOT_TRAINED_EXACT", [*nue, reader.H8S3, reader.NOT_TRAINED_EXACT]),
]:
    print(f"{label:<22} {len(rm.events(where=conds)):>8,} событий")

H8S3                      6,496 событий


+ NOT_TRAINED             4,198 событий


+ NOT_TRAINED_EXACT       4,497 событий


Обратите внимание на порядок: **`NOT_TRAINED_EXACT` оставляет больше событий**
(4 497 против 4 198), потому что он строже в другую сторону — исключает только
обучающую половину, а `NOT_TRAINED` выбрасывает весь датасет целиком, включая
валидацию и то, что отсеял `max_events`.

По умолчанию стоит консервативный `NOT_TRAINED`. Валидационные события модель не
училась предсказывать, но именно по ним выбиралась лучшая эпоха, так что их скоры
тоже смещены — просто слабее. Брать `NOT_TRAINED_EXACT` осмысленно, когда статистики
не хватает и вы готовы это смещение назвать вслух.

### Маски, которые нельзя выразить в SQL

Одна есть: у `mc_reco` многокластерное событие хранится по строке на кластер, и
каждый обломок несёт реконструкцию **целого** события рядом с хитами одного куска.
Признак обломка — `n_gt_sig_hits` — живёт в файле вероятностей, а не в базе, поэтому
фильтр применяется после чтения и подаётся отдельным аргументом `masks`.

Применённые маски всегда попадают в `chunk.meta`, чтобы ничего не срабатывало
незаметно.

In [11]:
rr = reader.open("mc_reco", checkpoint=CHECKPOINT)
cond = [reader.H8S3, "l.part_key LIKE 'part_2020_cl4_run103%'"]
print(f"без маски : {len(rr.events(where=cond)):,}")
print(f"с маской  : {len(rr.events(where=cond, masks=['whole_events'])):,}")

без маски : 6,040


с маской  : 6,040


## 5. Уровень 2: поток с хитами

`r.stream(...)` идёт по **частям** — это структурная единица источника: один прогон
детектора или один файл МК, со своими `ev_starts` и реконструкцией. Но часть не может
быть единицей памяти: от 202 событий в `mc_reco` до 3.7 млн (10.65 ГБ) в `exp_full`,
пять порядков. Поэтому мелкие части копятся, крупные режутся, а размер чанка задаётся
бюджетом `budget_mb`.

Что в чанке:

| поле | что |
|---|---|
| `chunk.events` | таблица на событие, как у `events()` |
| `chunk.hits` | таблица на хит: `event, q, t, x, y, z, channel, string, prob, is_sig` |
| `chunk.parts` | какие части попали в чанк |
| `chunk.meta` | применённые маски, число отрезков, сколько хитов прочитано и оставлено |

Колонка `event` в `hits` — это номер строки в `chunk.events`, а не `event_fk`.

In [12]:
for chunk in r.stream(where=[reader.H8S3, "l.part_key = 'part_s2020_c04_r0074'"],
                      budget_mb=128):
    print(chunk)
    print("meta:", chunk.meta)
    break
chunk.hits.head(4)

<Chunk 27,560 events, 2,089,421 hits, 2 part(s), 2 run(s)>
meta: {'masks': [], 'n_runs': 2, 'hits_read': 3392137, 'hits_kept': 2089421}


,event,q,t,x,y,z,channel,prob,string,is_sig
0,0,1.273893,-2237.297852,-58.828995,-6.829093,188.107254,210,0.002247,5,False
1,0,0.584728,-2180.887695,51.417477,31.247660,-217.630005,39,0.002193,1,False
2,0,1.014139,-2179.653320,51.531712,31.307596,-247.654221,37,0.002171,1,False
3,0,0.135783,-2111.258789,50.904324,30.977976,-82.528854,48,0.002245,1,False


Хиты привязываются к событиям через `chunk.hits.event` — это позиция в
`chunk.events`. Отсюда считается что угодно на событие.

In [13]:
sig = chunk.hits[chunk.hits.is_sig]
per_event = sig.groupby("event").agg(n_sig=("q", "size"),
                                     q_mean=("q", "mean"),
                                     t_span=("t", lambda v: v.max() - v.min()))
joined = chunk.events.join(per_event)
joined[["event_fk", "score", "n_sn_hits", "n_sig", "q_mean", "t_span"]].head()

,event_fk,score,n_sn_hits,n_sig,q_mean,t_span
0,663729294,0.001778,11,11,7.060658,508.264679
1,663729295,0.004234,10,10,2.471635,354.696289
2,663729296,0.000705,15,15,3.479169,393.938477
3,663729298,0.000874,28,28,5.254567,439.059570
4,663729304,0.001812,9,9,23.803421,362.947266


### `budget_mb` — единственная настройка памяти

Ограничивает хиты, удерживаемые за раз (около 41 байта на хит, измерено). Это
настройка **памяти**, а не отбора: какие события попадут в выборку, от неё не
зависит — на это есть тест.

In [14]:
cond = [reader.H8S3, "l.part_key = 'part_s2020_c04_r0074'"]
for budget in (16, 64, 256):
    chunks = list(r.stream(where=cond, budget_mb=budget))
    ids = np.sort(pd.concat([c.events for c in chunks]).event_fk.to_numpy())
    print(f"budget_mb={budget:>4}: {len(chunks)} чанк(ов), {len(ids):,} событий, "
          f"хиты в самом большом {max(len(c.hits) for c in chunks):,}")

budget_mb=  16: 8 чанк(ов), 27,560 событий, хиты в самом большом 367,236


budget_mb=  64: 2 чанк(ов), 27,560 событий, хиты в самом большом 1,105,063


budget_mb= 256: 1 чанк(ов), 27,560 событий, хиты в самом большом 2,089,421


### `with_hits=False` и `columns`

`with_hits=False` даёт поток без чтения хитов — то же, что `events()`, но по частям.
`columns` ограничивает колонки, которые тянутся из таблицы `predictions`.

In [15]:
n = sum(len(c.events) for c in
        r.stream(where=cond, with_hits=False, columns=["score"]))
print(f"{n:,} событий, хиты не читались")

27,560 событий, хиты не читались


### Полоска прогресса

Читалка **ничего не печатает**: это библиотека, и печать из неё портила бы логи,
ломала тесты и предполагала, что на том конце терминал. Вместо этого она сообщает
состояние через `progress=` — обратный вызов, получающий `Progress` со стадией и
счётчиками, — а решает, что с ним делать, вызывающий.

Стадии: `querying` (запрос стартует, строк ещё нет — на широком отборе это 7–20
секунд кажущейся тишины), `training` (сборка таблицы исключения обучения, ~3 с),
`reading`, `done`.

Итог для полоски даёт `count()`: один агрегатный запрос, секунды даже на базе в
94 млн строк.

In [16]:
from tqdm.auto import tqdm

cond = [reader.H8S3, "l.part_key = 'part_s2020_c04_r0074'"]
total = r.count(where=cond)
print("всего по count():", total)

with tqdm(total=total["events"], unit=" соб") as pbar:
    def report(state):
        pbar.set_description({"querying": "запрос", "done": "готово"}.get(
            state.stage, f"часть {state.parts}/{total['parts']}"))
        if state.events:
            pbar.n = state.events
        pbar.refresh()

    n_hits = sum(len(c.hits) for c in
                 r.stream(where=cond, budget_mb=128, progress=report))
print(f"прочитано хитов: {n_hits:,}")

всего по count(): {'events': 27560, 'parts': 1}


  0%|          | 0/27560 [00:00<?, ? соб/s]

прочитано хитов: 2,089,421


## 6. `verify` — сверка адресации

Пересчитывает `n_sn_hits` и `n_sn_strings` из HDF5 по `(group, part_key, local_idx)`
и сравнивает с тем, что записала в базу отдельная программа. Одной проверкой
покрываются ключ каталога, строка HDF5, выравнивание файла вероятностей и
соглашение о пороге.

Это не украшение: именно она поймала `>=` вместо `>` в маске сигнального хита —
одно событие из 20 000, у которого вероятность ровно `float32(0.8)`.

In [17]:
r.verify(chunk)

,check,events,mismatched
0,n_sn_hits,27560,0
1,n_sn_strings,27560,0


## 7. Вспомогательное

`part_map()` — карта частей источника с диапазонами `event_fk`. Потоку она не нужна
(строки и так приходят сгруппированными), поэтому строится только по запросу: для
`mc_merged` запрос к каталогу занимает около 27 секунд.

`check_against_converter_config()` — сверяет имена колонок `reco_prty` с конфигом
конвертера ROOT→HDF5. Вызывайте, если подозреваете, что продукция изменилась.

In [18]:
print(reader.check_against_converter_config("/home/albert/Baikal2025"))
pm = r.part_map()
print(f"\nчастей в exp_reco: {len(pm):,}")
pm.head(3)

{'exp_reco': 25, 'mc_reco': 31}



частей в exp_reco: 370


,data_class,part_key,n,lo,hi,group
0,exp_reco,part_s2020_c03_r0009,257171,660157401,660414571,exp_reco
1,exp_reco,part_s2020_c06_r0025,163037,666712530,666875566,exp_reco
2,exp_reco,part_s2020_c04_r0126,132287,664677626,664809912,exp_reco


## 8. Сколько это стоит

Дальше — замеры. Общая функция: прогоняет поток целиком и считает время, число
отрезков чтения и **переплату** — отношение прочитанных хитов к оставленным.

Переплата возникает из-за сжатия: `raw/data` лежит gzip-чанками по 72 461 значению
на колонку, и меньше одного чанка прочитать нельзя. Если нужно 0.1 % событий части,
за них всё равно платится чанками.

In [19]:
def timed(source, where, budget_mb=256, masks=(), with_hits=True):
    rd = reader.open(source, checkpoint=CHECKPOINT)
    t0 = time.time(); ev = hits = read = runs = chunks = 0
    for c in rd.stream(where=where, masks=masks, budget_mb=budget_mb,
                       with_hits=with_hits):
        ev += len(c.events); hits += len(c.hits)
        read += c.meta.get("hits_read", 0); runs += c.meta.get("n_runs", 0)
        chunks += 1
    dt = time.time() - t0
    return dict(events=ev, hits=hits, chunks=chunks, runs=runs,
                overpay=round(read / max(hits, 1), 1), seconds=round(dt, 1),
                us_per_event=round(dt / max(ev, 1) * 1e6, 1))

### Как время зависит от источника

In [20]:
rows = []
for src, cond in [
    ("exp_reco",  [reader.H8S3, "l.part_key = 'part_s2020_c04_r0074'"]),
    ("mc_merged", [reader.H8S3, "l.part_key = 'part_10065'"]),
    ("mc_reco",   [reader.H8S3, "l.part_key LIKE 'part_2020_cl4_run103%'"]),
    ("exp_full",  [reader.H8S3, "l.part_key = 'part_s2020_c05_r0022'",
                   "p.score > 0.5"]),
]:
    rows.append({"source": src, **timed(src, cond)})
pd.DataFrame(rows)

,source,events,hits,chunks,runs,overpay,seconds,us_per_event
0,exp_reco,27560,2089421,1,1,1.6,1.3,48.7
1,mc_merged,3203,241124,1,1,8.5,1.0,298.5
2,mc_reco,6040,451290,1,56,1.8,1.8,295.7
3,exp_full,994,67870,1,797,97.2,54.1,54405.3


Разброс на два порядка, и причина в каждом случае своя.

- **`exp_reco`** — лучший случай: 63 % событий части проходят h8s3, отбор плотный,
  один отрезок чтения, переплата 1.6×.
- **`mc_merged`** — из части h8s3 проходит около 10 %, отбор разрежен, переплата 8.5×.
- **`mc_reco`** — 56 отрезков, потому что части крошечные (медиана 202 события) и в
  один чанк их собирается много.
- **`exp_full`** — худший случай: подходящих событий там около 1 %, отбор размазан по
  части из 5.9 млн событий, отсюда 797 отрезков и переплата почти 100×. Это цена
  gzip-чанков, а не выбор стратегии: читать меньше физически нельзя.

### Как время зависит от размера выборки

In [21]:
parts8 = ["part_s2020_c04_r0074", "part_s2020_c04_r0080", "part_s2020_c04_r0108",
          "part_s2020_c04_r0130", "part_s2020_c05_r0196", "part_s2020_c05_r0337",
          "part_s2020_c06_r0100", "part_s2020_c06_r0119"]
rows = []
for k in (1, 2, 4, 8):
    listed = ", ".join(f"'{p}'" for p in parts8[:k])
    rows.append({"parts": k, **timed("exp_reco",
                 [reader.H8S3, f"l.part_key IN ({listed})"])})
pd.DataFrame(rows)

,parts,events,hits,chunks,runs,overpay,seconds,us_per_event
0,1,27560,2089421,1,1,1.6,1.4,50.9
1,2,55437,4237182,1,2,1.6,2.0,36.0
2,4,102434,7757696,2,4,1.6,3.5,34.3
3,8,121586,9547934,2,6,1.8,4.0,33.1


Время растёт **медленнее**, чем объём: на событие выходит 59 мкс при одной части и
37 мкс при четырёх. Причина — постоянная стоимость около секунды: запуск потокового
запроса DuckDB плюс открытие файлов. Она размазывается по мере роста выборки.

Практический вывод: много мелких чтений дороже одного крупного. Если нужны десять
частей, берите их одним `where`, а не десятью вызовами.

### Как время зависит от фильтров

In [22]:
p10065 = "l.part_key = 'part_10065'"
rows = []
for label, conds, hits_flag in [
    ("H8S3",                 [p10065, reader.H8S3], True),
    ("+ NOT_TRAINED",        [p10065, reader.H8S3, reader.NOT_TRAINED], True),
    ("+ NOT_TRAINED_EXACT",  [p10065, reader.H8S3, reader.NOT_TRAINED_EXACT], True),
    ("+ score > 0.9",        [p10065, reader.H8S3, reader.NOT_TRAINED,
                              "p.score > 0.9"], True),
    ("NOT_TRAINED, без хитов", [p10065, reader.H8S3, reader.NOT_TRAINED], False),
]:
    rows.append({"filter": label,
                 **timed("mc_merged", conds, with_hits=hits_flag)})
pd.DataFrame(rows)

,filter,events,hits,chunks,runs,overpay,seconds,us_per_event
0,H8S3,3203,241124,1,1,8.5,1.0,317.9
1,+ NOT_TRAINED,3203,241124,1,1,8.5,4.2,1305.2
2,+ NOT_TRAINED_EXACT,3203,241124,1,1,8.5,4.1,1267.9
3,+ score > 0.9,1,70,1,1,1.0,3.8,3829996.8
4,"NOT_TRAINED, без хитов",3203,0,1,0,0.0,3.9,1209.5


Главное здесь: **`NOT_TRAINED` стоит около 3.5 секунд, и это постоянная плата**, а
не плата за событие. Она уходит на сборку таблицы тождества из NPY-датасета —
5 256 496 строк — и её регистрацию в DuckDB. На большой выборке эти три секунды
незаметны, на однократном чтении одной части они втрое увеличивают время.

Строгий вариант `NOT_TRAINED_EXACT` не дороже: разбиение воспроизводится двумя
вызовами `numpy`.

`with_hits=False` здесь экономит мало именно потому, что правит постоянная
стоимость, а не чтение хитов.

### Как переплата зависит от плотности отбора

In [23]:
rows = []
for thr in (0.0, 0.5, 0.9, 0.99):
    rows.append({"score >": thr, **timed("exp_reco",
        [reader.H8S3, "l.part_key = 'part_s2020_c04_r0074'", f"p.score > {thr}"])})
pd.DataFrame(rows)

,score >,events,hits,chunks,runs,overpay,seconds,us_per_event
0,0.00,27560,2089421,1,1,1.6,1.3,46.8
1,0.50,230,16898,1,4,184.8,1.1,4658.8
2,0.90,49,3689,1,16,287.6,0.9,18497.3
3,0.99,7,553,1,6,74.2,0.8,110383.0


Чем уже отбор, тем больше лишнего приходится поднять с диска: от 1.6× при полном
охвате до почти 300× при `score > 0.9`. Абсолютное время при этом **падает** —
событий-то меньше, — но на событие растёт на порядки.

Отсюда рабочий приём: если нужны редкие события и много признаков по ним, сначала
возьмите `events()` без хитов, отберите `event_fk`, и лишь потом читайте хиты
отдельным запросом по этому списку. Один проход по хитам ради 0.1 % событий —
самый дорогой способ.

## 9. Свой цикл вместо встроенной агрегации

Готовой `aggregate` в читалке нет намеренно: это цикл в три строки, а обёртка
задавала бы соглашения раньше, чем понятно, какие нужны. Существенное свойство —
что хиты нигде не накапливаются — сохраняется и без неё.

In [24]:
def features(events, hits):
    sig = hits[hits.is_sig]
    agg = sig.groupby("event").agg(n_sig=("q", "size"), q_mean=("q", "mean"),
                                   z_span=("z", lambda v: v.max() - v.min()))
    return events[["event_fk", "score"]].join(agg)

stats = pd.concat([features(c.events, c.hits) for c in
                   r.stream(where=[reader.H8S3, "e.cluster = 4",
                                   "p.score > 0.8"], budget_mb=128)],
                  ignore_index=True)
print(f"{len(stats):,} событий; таблица {stats.memory_usage(deep=True).sum()/1e6:.1f} МБ")
stats.describe().round(3)

5,507 событий; таблица 0.2 МБ


,event_fk,score,n_sig,q_mean,z_span
count,5.507000e+03,5507.000,5507.000,5507.000,5507.000
mean,6.640539e+08,0.920,13.591,45.687,100.748
std,6.671191e+05,0.063,5.516,122.233,65.357
min,6.627005e+08,0.800,8.000,1.298,15.360
25%,6.634758e+08,0.862,10.000,5.034,60.027
50%,6.642323e+08,0.932,12.000,11.915,75.123
75%,6.646335e+08,0.984,16.000,45.206,105.724
max,6.649731e+08,0.996,106.000,4808.109,450.661


Сохранять результат — ваше дело: `stats.to_parquet(...)`. Библиотека на диск не
пишет ничего.

## Итог

- `runs()`, `sources()`, `unresolved()` — что есть и что сломано;
- `r.events(where=...)` — таблица на событие, хиты не читаются;
- `r.stream(where=..., masks=..., budget_mb=..., with_hits=...)` — поток по частям;
- `r.verify(chunk)` — сверка адресации, запускайте на новых выборках;
- фильтры складываются списком: SQL по псевдонимам `p`/`e`/`l` плюс именованные
  предикаты.

Что стоит помнить о времени: постоянная стоимость около секунды на запрос и ещё три
секунды на `NOT_TRAINED`; переплата за чтение растёт по мере сужения отбора; много
мелких чтений дороже одного крупного.